# Sales by training hours: Jonckheere-Terpstra trend test

**Jonckheere-Terpstra Test** · a worked example from the StatNex teaching guide

**The story.** A company tracked the monthly sales of reps who had 0, 4, 8 or 16 hours of sales training, five reps per group.

**The question.** Do sales rise steadily as training hours increase?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/jonckheere_terpstra.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
0 h,4 h,8 h,16 h
20.5,21.3,22.3,23.5
19.4,21.0,21.2,22.1
20.3,20.9,21.7,21.6
19.8,21.7,21.5,22.0
19.6,20.9,22.7,21.8

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per level, in the order the trend is expected to follow. Text and blank
# cells are skipped, so levels may differ in size.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["0 h", "4 h", "8 h", "16 h"]
samples = [pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float) for name in levels]
sizes = np.array([len(s) for s in samples], dtype=float)
print(pd.DataFrame({"n": sizes.astype(int)}, index=levels).T)

## Pair counts

In [ ]:
# For each pair of levels i < j, count the pairs of values where the later level's value
# is larger (increase) or smaller (decrease). If the levels made no difference, a pair
# would be as likely to go up as down. Equal values count in neither table; the test
# below adds half of each tied pair to both directions.
k = len(levels)
increase = pd.DataFrame(np.nan, index=levels[:-1], columns=levels[1:])
decrease = pd.DataFrame(np.nan, index=levels[:-1], columns=levels[1:])
for i in range(k):
    for j in range(i + 1, k):
        later, earlier = samples[j][:, None], samples[i][None, :]
        increase.iloc[i, j - 1] = (later > earlier).sum()
        decrease.iloc[i, j - 1] = (later < earlier).sum()
for name, table in (("Increasing trend", increase), ("Decreasing trend", decrease)):
    print(f"\n{name} (total {table.sum().sum():.0f})")
    print(table)

## Jonckheere-Terpstra test

In [ ]:
# J for each direction is its pair count plus half of the tied pairs (Hollander and
# Wolfe), so the two rows mirror each other. J is standardised with its null mean and
# its variance corrected for ties (Lehmann), as Stata's nptrend does, giving z; P is
# one-sided, in the direction of that row: read the row your design predicted before
# seeing the data. No scipy function does this test, so it is written out.
total = sizes.sum()
expected = (total**2 - (sizes**2).sum()) / 4
# t: the size of each group of equal values, pooled over all levels.
_, t = np.unique(np.concatenate(samples), return_counts=True)
t, n = t.astype(float), sizes
variance = (total * (total - 1) * (2 * total + 5) - (n * (n - 1) * (2 * n + 5)).sum()
            - (t * (t - 1) * (2 * t + 5)).sum()) / 72
if total > 2:
    variance += ((n * (n - 1) * (n - 2)).sum() * (t * (t - 1) * (t - 2)).sum()
                 / (36 * total * (total - 1) * (total - 2)))
variance += (n * (n - 1)).sum() * (t * (t - 1)).sum() / (8 * total * (total - 1))
pairs = sum(sizes[i] * sizes[j] for i in range(k) for j in range(i + 1, k))
tied = pairs - increase.sum().sum() - decrease.sum().sum()
rows = []
for name, table in (("increasing trend", increase), ("decreasing trend", decrease)):
    count = table.sum().sum()
    z = (count + tied / 2 - expected) / np.sqrt(variance)
    rows.append([count, z, stats.norm.sf(z)])
test = pd.DataFrame(rows, index=["increasing trend", "decreasing trend"],
                    columns=["pairs", "z", "P (one-sided)"])
print()
print(test)

## What the result says

The Jonckheere-Terpstra test asks whether values rise step by step across ordered groups. Here the groups are sales reps with 0, 4, 8 and 16 hours of training, 5 reps in each.

The test compares each rep with every rep who had more training. In 136 of these pairs the rep with more training sold more, and in only 13 pairs that rep sold less. For the increasing trend the statistic z is 4.14 and the P value is 0.00002. The P value is the chance of a result this strong if there were no trend, so the trend is clear.

Most of the rise comes early. Between 8 and 16 hours, 16 pairs go up but 9 go down. Sales rise with training hours, but the reps chose their own training, so this does not prove that training causes the rise.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.